In [ ]:
# ============================================================
# CELL 1 — COMPLETE DQN-VSSI-EWMA CODE from IC calibration, DQN training and OC computations
# ============================================================

import os
import copy
import math
import random
import numpy as np
import pandas as pd

from dataclasses import dataclass
from statistics import NormalDist
from collections import deque
from joblib import Parallel, delayed

import torch
import torch.nn as nn
import torch.optim as optim
import torch.nn.functional as F
import time
notebook_start = time.perf_counter()

# ============================================================
# GLOBAL SETTINGS
# ============================================================

N_JOBS = min(8, max(1, os.cpu_count() - 1))


def set_global_seed(seed=12345):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)

    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)


# ============================================================
# BASIC EWMA FUNCTIONS
# ============================================================

def normal_ppf(q):
    return NormalDist().inv_cdf(q)


def warning_coefficient_from_p0(p0):
    return normal_ppf((1.0 + p0) / 2.0)


def ewma_limits(L, p0, lam):
    sigma_z = math.sqrt(lam / (2.0 - lam))
    W = warning_coefficient_from_p0(p0)

    H = L * sigma_z
    HW = W * sigma_z

    return H, HW, W, sigma_z


def update_ewma(z_prev, y_t, lam):
    return lam * y_t + (1.0 - lam) * z_prev


def make_state(z, z_prev, prev_n, prev_h, H, HW, cfg):
    n_max = max(cfg.n_values)
    h_max = max(cfg.h_values)

    closeness = abs(z) / H
    movement = abs(z - z_prev) / H
    prev_n_scaled = prev_n / n_max
    prev_h_scaled = prev_h / h_max
    warning_indicator = 1.0 if abs(z) > HW else 0.0

    return np.array(
        [
            closeness,
            movement,
            prev_n_scaled,
            prev_h_scaled,
            warning_indicator
        ],
        dtype=np.float32
    )


def split_reps(n_rep, n_jobs):
    n_jobs = min(n_jobs, n_rep)

    base = n_rep // n_jobs
    rem = n_rep % n_jobs

    chunks = []
    for j in range(n_jobs):
        chunks.append(base + (1 if j < rem else 0))

    return chunks


# ============================================================
# CONFIG
# ============================================================

@dataclass
class Config:
    # Process and EWMA
    mu0: float = 0.0
    sigma0: float = 1.0
    lam: float = 0.2
    p0: float = 0.5
    L: float = 2.8
    # Any initial value can be used for L; L is automatically recalibrated before training.

    # Action space
    n_values: tuple = tuple(range(1, 11))
    h_values: tuple = (0.25, 0.50, 0.75, 1.00, 1.25, 1.50, 2.00)

    # IC design targets
    target_arl0: float = 370.4
    target_avg_h: float = 1.0
    target_avg_n: float = 3.5

    # Acceptance tolerances
    avg_h_tol: float = 0.03
    avg_n_tol: float = 0.03

    # Slopes of the VSSI structural target functions.
    # n_target increases with risk; h_target decreases with risk.
    n_shape_slope: float = 3.0
    h_shape_slope: float = 1.0
    
    # These are filled after pre-training IC rho calibration.
    mean_rho_ic: float = 0.0
    
    n_safe_target: float = 2.0
    n_warning_target: float = 5.0
    
    h_safe_target: float = 1.5
    h_warning_target: float = 0.5
    
    # Risk-adaptive shaping weights.
    # These encourage a real VSSI pattern:
    # as risk increases, n should increase and h should decrease.
    c_shape_n: float = 1.0
    c_shape_h: float = 1.0
    
    # IC average-resource reward coefficients.
    # These control the IC average constraints, not individual actions.
    c_ic_n_running: float = 5.0
    c_ic_h_running: float = 5.0
    c_ic_n_terminal: float = 300.0
    c_ic_h_terminal: float = 300.0
    
    # OC and false-alarm reward coefficients.
    # The OC delay cost is risk-weighted in the step function.
    c_delay: float = 2.2
    c_false: float = 100.0
    detect_reward: float = 50.0
     
    # Minimum OC delay pressure in very safe states.
    oc_risk_floor: float = 0.10

    # Training mixture
    p_ic_train: float = 0.50

    # Episode cap
    max_steps: int = 10000


def create_actions(cfg):
    return [(n, h) for n in cfg.n_values for h in cfg.h_values]


def validate_config(cfg):
    W = warning_coefficient_from_p0(cfg.p0)

    assert 0.0 < cfg.p0 < 1.0
    assert 0.0 < cfg.lam < 1.0
    assert cfg.L > W
    assert min(cfg.n_values) >= 1
    assert min(cfg.h_values) > 0
    assert cfg.target_arl0 > 0
    assert cfg.target_avg_h > 0
    assert cfg.target_avg_n > 0

    return True


# ============================================================
# PRE-TRAINING DIRECT L CALIBRATION
# This does not use the DQN policy because, under IC,
# the standardized EWMA sequence is independent of n_t and h_t.
# ============================================================

def simulate_direct_ic_rl(L, cfg, rng):
    H, HW, W, sigma_z = ewma_limits(L, cfg.p0, cfg.lam)

    z = rng.normal(0.0, sigma_z)

    for step in range(1, cfg.max_steps + 1):
        y_t = rng.normal(loc=0.0, scale=1.0)
        z = update_ewma(z, y_t, cfg.lam)

        if abs(z) > H:
            return step

    return cfg.max_steps


def direct_ic_chunk(L, cfg, n_rep, seed):
    rng = np.random.default_rng(seed)
    rls = np.zeros(n_rep)

    for i in range(n_rep):
        rls[i] = simulate_direct_ic_rl(L, cfg, rng)

    return rls


def estimate_direct_arl0(
    L,
    cfg,
    n_rep=100000,
    n_jobs=N_JOBS,
    base_seed=98765
):
    chunks = split_reps(n_rep, n_jobs)

    results = Parallel(
        n_jobs=len(chunks),
        backend="loky"
    )(
        delayed(direct_ic_chunk)(
            L,
            copy.deepcopy(cfg),
            chunks[j],
            base_seed + j
        )
        for j in range(len(chunks))
    )

    rls = np.concatenate(results)

    return float(np.mean(rls))


def calibrate_L_direct(
    cfg,
    target_arl0=370.4,
    low=None,
    high=5.0,
    n_iter=20,
    n_rep=10000
):
    W = warning_coefficient_from_p0(cfg.p0)

    if low is None:
        low = W + 0.05

    best_L = None
    best_arl = None
    best_error = np.inf

    for it in range(n_iter):
        mid = 0.5 * (low + high)

        arl0 = estimate_direct_arl0(
            L=mid,
            cfg=cfg,
            n_rep=n_rep,
            n_jobs=N_JOBS,
            base_seed=98765
        )

        error = abs(arl0 - target_arl0)

        if error < best_error:
            best_error = error
            best_L = mid
            best_arl = arl0

        print(
            f"Direct L calibration iter {it+1:02d}: "
            f"L={mid:.5f}, ARL0={arl0:.4f}"
        )

        if arl0 < target_arl0:
            low = mid
        else:
            high = mid

    cfg.L = best_L

    print("\nPre-training calibrated L")
    print(f"L       = {best_L:.5f}")
    print(f"ARL0    = {best_arl:.4f}")

    return best_L

def simulate_direct_ic_rho_sum(L, cfg, rng):
    H, HW, W, sigma_z = ewma_limits(L, cfg.p0, cfg.lam)

    z = rng.normal(loc=0.0, scale=sigma_z)
    rho_sum = 0.0
    steps = 0

    for step in range(1, cfg.max_steps + 1):
        rho = min(abs(z) / H, 1.0)

        rho_sum += rho
        steps += 1

        y_t = rng.normal(loc=0.0, scale=1.0)
        z = update_ewma(z, y_t, cfg.lam)

        if abs(z) > H:
            break

    return rho_sum, steps


def direct_ic_rho_chunk(L, cfg, n_rep, seed):
    rng = np.random.default_rng(seed)

    rho_sum_total = 0.0
    steps_total = 0

    for i in range(n_rep):
        rho_sum, steps = simulate_direct_ic_rho_sum(
            L=L,
            cfg=cfg,
            rng=rng
        )

        rho_sum_total += rho_sum
        steps_total += steps

    return rho_sum_total, steps_total


def estimate_direct_mean_rho_ic(
    L,
    cfg,
    n_rep=10000,
    n_jobs=N_JOBS,
    base_seed=77777
):
    chunks = split_reps(n_rep, n_jobs)

    results = Parallel(
        n_jobs=len(chunks),
        backend="loky"
    )(
        delayed(direct_ic_rho_chunk)(
            L,
            copy.deepcopy(cfg),
            chunks[j],
            base_seed + j
        )
        for j in range(len(chunks))
    )

    rho_sum_total = sum(r[0] for r in results)
    steps_total = sum(r[1] for r in results)

    return rho_sum_total / steps_total


def set_vssi_structural_targets_from_ic_rho(cfg):
    mean_rho = cfg.mean_rho_ic

    cfg.n_safe_target = (
        cfg.target_avg_n
        - cfg.n_shape_slope * mean_rho
    )

    cfg.n_warning_target = (
        cfg.n_safe_target
        + cfg.n_shape_slope
    )

    cfg.h_safe_target = (
        cfg.target_avg_h
        + cfg.h_shape_slope * mean_rho
    )

    cfg.h_warning_target = (
        cfg.h_safe_target
        - cfg.h_shape_slope
    )

    print("\nIC-rho-calibrated VSSI structural targets")
    print(f"mean rho IC          = {cfg.mean_rho_ic:.5f}")
    print(f"n_safe_target        = {cfg.n_safe_target:.5f}")
    print(f"n_warning_target     = {cfg.n_warning_target:.5f}")
    print(f"h_safe_target        = {cfg.h_safe_target:.5f}")
    print(f"h_warning_target     = {cfg.h_warning_target:.5f}")
    
# ============================================================
# ENVIRONMENT WITH RUNNING-AVERAGE AND TERMINAL-AVERAGE IC REWARD
# ============================================================

class EWMAVSSIEnv:
    def __init__(
        self,
        cfg,
        actions,
        delta_choices=(0.25, 0.5, 0.75, 1.0, 1.5, 2.0)
    ):
        self.cfg = cfg
        self.actions = actions
        self.delta_choices = delta_choices
        self.reset()

    def reset(self):
        # --- compute limits FIRST, so self.sigma_z is available ---
        self.H, self.HW, self.W, self.sigma_z = ewma_limits(
            self.cfg.L, self.cfg.p0, self.cfg.lam
        )
    
        # --- now self.sigma_z exists, safe to use ---
        self.z = np.random.normal(0.0, self.sigma_z)
        self.z_prev = self.z
    
        #self.prev_n = min(self.cfg.n_values)
        #self.prev_h = max(self.cfg.h_values)
        self.prev_n = self.cfg.target_avg_n
        self.prev_h = self.cfg.target_avg_h
    
        self.total_n = 0.0
        self.total_h = 0.0
        self.total_steps = 0
    
        if np.random.rand() < self.cfg.p_ic_train:
            self.delta = 0.0
            self.is_ic_episode = True
        else:
            self.delta = float(np.random.choice(self.delta_choices))
            self.is_ic_episode = False
    
        return make_state(
            self.z, self.z_prev, self.prev_n, self.prev_h,
            self.H, self.HW, self.cfg
        )

    def step(self, action_index):
        n_t, h_t = self.actions[action_index]
    
        # Risk before taking the action.
        # This is the evidence level used to decide whether the chart is in a
        # safe, intermediate, or warning-like state.
        rho = min(abs(self.z) / self.H, 1.0)
    
        # VSSI adaptive targets.
        # These are not IC average constraints. They only guide the policy toward
        # a logical adaptive pattern.
        n_target_rho = (
            self.cfg.n_safe_target
            + rho * (self.cfg.n_warning_target - self.cfg.n_safe_target)
        )
    
        h_target_rho = (
            self.cfg.h_safe_target
            - rho * (self.cfg.h_safe_target - self.cfg.h_warning_target)
        )
    
        current_delta = 0.0 if self.is_ic_episode else self.delta
    
        y_t = np.random.normal(
            loc=math.sqrt(n_t) * current_delta,
            scale=1.0
        )
    
        self.z_prev = self.z
        self.z = update_ewma(self.z_prev, y_t, self.cfg.lam)
    
        self.total_n += n_t
        self.total_h += h_t
        self.total_steps += 1
    
        signal = abs(self.z) > self.H
    
        reward = 0.0
    
        # ------------------------------------------------------------
        # VSSI structural shaping.
        # This prevents the degenerate policy "always use very small h".
        # It encourages:
        # safe region: smaller n and longer h
        # warning region: larger n and shorter h
        # ------------------------------------------------------------
        n_shape_error = (
            n_t - n_target_rho
        ) / max(
            abs(self.cfg.n_warning_target - self.cfg.n_safe_target),
            1e-12
        )
    
        h_shape_error = (
            h_t - h_target_rho
        ) / max(
            abs(self.cfg.h_safe_target - self.cfg.h_warning_target),
            1e-12
        )
    
        reward -= self.cfg.c_shape_n * (n_shape_error ** 2)
        reward -= self.cfg.c_shape_h * (h_shape_error ** 2)
    
        # ------------------------------------------------------------
        # IC reward:
        # Penalize running-average IC resource deviations.
        # This controls average n0 and average h0.
        # It does NOT punish individual actions for being different from n0 or h0.
        # ------------------------------------------------------------
        if self.is_ic_episode:
            running_avg_n = self.total_n / self.total_steps
            running_avg_h = self.total_h / self.total_steps
    
            n_error_running = (
                running_avg_n - self.cfg.target_avg_n
            ) / self.cfg.target_avg_n
    
            h_error_running = (
                running_avg_h - self.cfg.target_avg_h
            ) / self.cfg.target_avg_h
    
            reward -= self.cfg.c_ic_n_running * (n_error_running ** 2)
            reward -= self.cfg.c_ic_h_running * (h_error_running ** 2)
    
        # ------------------------------------------------------------
        # OC reward:
        # Penalize calendar delay more strongly when the EWMA statistic is
        # closer to the control limit. This preserves the adaptive VSSI logic.
        # ------------------------------------------------------------
        if not self.is_ic_episode:
            oc_delay_weight = self.cfg.oc_risk_floor + rho
            #reward -= self.cfg.c_delay * oc_delay_weight * h_t
            effective_delay = h_t / math.sqrt(n_t / self.cfg.target_avg_n)
            reward -= self.cfg.c_delay * oc_delay_weight * effective_delay

    
        done = False
    
        if signal:
            if self.is_ic_episode:
                reward -= self.cfg.c_false
            else:
                reward += self.cfg.detect_reward
    
            done = True
    
        if self.total_steps >= self.cfg.max_steps:
            done = True
    
        # ------------------------------------------------------------
        # Terminal IC episode-average resource penalty.
        # ------------------------------------------------------------
        if done and self.is_ic_episode and self.total_steps > 0:
            episode_avg_n = self.total_n / self.total_steps
            episode_avg_h = self.total_h / self.total_steps
    
            n_error_episode = (
                episode_avg_n - self.cfg.target_avg_n
            ) / self.cfg.target_avg_n
    
            h_error_episode = (
                episode_avg_h - self.cfg.target_avg_h
            ) / self.cfg.target_avg_h
    
            reward -= self.cfg.c_ic_n_terminal * (n_error_episode ** 2)
            reward -= self.cfg.c_ic_h_terminal * (h_error_episode ** 2)
    
        self.prev_n = n_t
        self.prev_h = h_t
    
        next_state = make_state(
            self.z,
            self.z_prev,
            self.prev_n,
            self.prev_h,
            self.H,
            self.HW,
            self.cfg
        )
    
        info = {
            "z": self.z,
            "n": n_t,
            "h": h_t,
            "signal": signal,
            "delta": self.delta,
            "is_ic_episode": self.is_ic_episode,
            "rho": rho
        }
    
        return next_state, reward, done, info

# ============================================================
# DQN NETWORK AND TRAINING
# ============================================================

class DQN(nn.Module):
    def __init__(self, state_dim, n_actions):
        super().__init__()

        self.net = nn.Sequential(
            nn.Linear(state_dim, 128),
            nn.ReLU(),
            nn.Linear(128, 128),
            nn.ReLU(),
            nn.Linear(128, n_actions)
        )

    def forward(self, x):
        return self.net(x)


class ReplayBuffer:
    def __init__(self, capacity):
        self.buffer = deque(maxlen=capacity)

    def push(self, state, action, reward, next_state, done):
        self.buffer.append(
            (
                np.asarray(state, dtype=np.float32),
                int(action),
                float(reward),
                np.asarray(next_state, dtype=np.float32),
                float(done)
            )
        )

    def sample(self, batch_size):
        batch = random.sample(self.buffer, batch_size)

        states, actions, rewards, next_states, dones = zip(*batch)

        return (
            np.asarray(states, dtype=np.float32),
            np.asarray(actions, dtype=np.int64),
            np.asarray(rewards, dtype=np.float32),
            np.asarray(next_states, dtype=np.float32),
            np.asarray(dones, dtype=np.float32)
        )

    def __len__(self):
        return len(self.buffer)


def select_action_epsilon_greedy(model, state, n_actions, epsilon, device):
    if random.random() < epsilon:
        return random.randrange(n_actions)

    with torch.no_grad():
        state_t = torch.tensor(
            state,
            dtype=torch.float32,
            device=device
        ).unsqueeze(0)

        q_values = model(state_t)

        return int(torch.argmax(q_values, dim=1).item())


def train_dqn(
    cfg,
    actions,
    n_episodes=8000,
    model=None,
    epsilon_start=1.0,
    epsilon_end=0.05,
    epsilon_decay=0.995,
    gamma=0.995,
    lr=1e-3,
    batch_size=64,
    buffer_size=50000,
    target_update=100,
    verbose=True
):
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

    env = EWMAVSSIEnv(cfg=cfg, actions=actions)

    state_dim = len(env.reset())
    n_actions = len(actions)

    if model is None:
        policy_net = DQN(state_dim, n_actions).to(device)
    else:
        policy_net = model.to(device)

    target_net = DQN(state_dim, n_actions).to(device)
    target_net.load_state_dict(policy_net.state_dict())
    target_net.eval()

    optimizer = optim.Adam(policy_net.parameters(), lr=lr)

    replay_buffer = ReplayBuffer(buffer_size)

    epsilon = epsilon_start
    episode_rewards = []

    for episode in range(1, n_episodes + 1):
        state = env.reset()
        total_reward = 0.0
        done = False

        while not done:
            action = select_action_epsilon_greedy(
                model=policy_net,
                state=state,
                n_actions=n_actions,
                epsilon=epsilon,
                device=device
            )

            next_state, reward, done, info = env.step(action)

            replay_buffer.push(
                state,
                action,
                reward,
                next_state,
                done
            )

            state = next_state
            total_reward += reward

            if len(replay_buffer) >= batch_size:
                states, actions_b, rewards_b, next_states, dones = replay_buffer.sample(
                    batch_size
                )

                states_t = torch.tensor(states, dtype=torch.float32, device=device)
                actions_t = torch.tensor(actions_b, dtype=torch.int64, device=device).unsqueeze(1)
                rewards_t = torch.tensor(rewards_b, dtype=torch.float32, device=device).unsqueeze(1)
                next_states_t = torch.tensor(next_states, dtype=torch.float32, device=device)
                dones_t = torch.tensor(dones, dtype=torch.float32, device=device).unsqueeze(1)

                q_values = policy_net(states_t).gather(1, actions_t)

                with torch.no_grad():
                    next_q_values = target_net(next_states_t).max(dim=1, keepdim=True)[0]
                    target_q_values = rewards_t + gamma * next_q_values * (1.0 - dones_t)

                loss = F.mse_loss(q_values, target_q_values)

                optimizer.zero_grad()
                loss.backward()
                torch.nn.utils.clip_grad_norm_(policy_net.parameters(), 5.0)
                optimizer.step()

        episode_rewards.append(total_reward)

        epsilon = max(epsilon_end, epsilon * epsilon_decay)

        if episode % target_update == 0:
            target_net.load_state_dict(policy_net.state_dict())

        if verbose and episode % 100 == 0:
            print(
                f"Episode {episode:5d} | "
                f"epsilon={epsilon:.3f} | "
                f"mean reward={np.mean(episode_rewards[-100:]):.3f}"
            )

    return policy_net.cpu(), episode_rewards


def train_dqn_two_stage_best_checkpoint(
    cfg,
    actions,
    initial_train_episodes=20000,
    fine_tune_episodes=10000,
    check_interval=1000,
    ic_check_rep=10000,
    oc_check_rep=5000,
    validation_shifts=(0.25, 0.50, 0.75, 1.00, 1.50, 2.00),
    ic_check_jobs=N_JOBS,
    gamma=0.995,
    batch_size=64,
    buffer_size=50000,
    target_update=100,
    verbose=True
):
    """
    Two-stage DQN training with best-checkpoint selection.

    Stage 1 matches the original initial training:
        episodes = initial_train_episodes, epsilon_start = 1.0, lr = 1e-3
    Stage 2 matches the original fine-tuning:
        episodes = fine_tune_episodes, epsilon_start = 0.20, lr = 5e-4

    Checkpoints are evaluated every `check_interval` episodes. Training does
    not restart inside a stage after checkpoint evaluation. The final returned
    model is the IC-feasible checkpoint with the smallest mean validation ATS1.
    """
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

    env = EWMAVSSIEnv(cfg=cfg, actions=actions)
    state_dim = len(env.reset())
    n_actions = len(actions)

    policy_net = DQN(state_dim, n_actions).to(device)

    checkpoint_history = []
    all_rewards = []

    best_feasible_state = None
    best_feasible_metrics = None
    best_feasible_episode = None
    best_feasible_stage = None
    best_feasible_mean_ats1 = np.inf
    best_feasible_mean_arl1 = np.inf

    best_constraint_state = None
    best_constraint_metrics = None
    best_constraint_episode = None
    best_constraint_stage = None
    best_constraint_score = np.inf

    global_episode = 0

    stages = [
        {
            "stage_name": "initial",
            "n_episodes": initial_train_episodes,
            "epsilon_start": 1.0,
            "epsilon_end": 0.05,
            "epsilon_decay": 0.995,
            "lr": 1e-3
        },
        {
            "stage_name": "fine_tuning",
            "n_episodes": fine_tune_episodes,
            "epsilon_start": 0.20,
            "epsilon_end": 0.05,
            "epsilon_decay": 0.995,
            "lr": 5e-4
        }
    ]

    for stage in stages:
        stage_name = stage["stage_name"]
        n_episodes = stage["n_episodes"]
        epsilon = stage["epsilon_start"]
        epsilon_end = stage["epsilon_end"]
        epsilon_decay = stage["epsilon_decay"]
        lr = stage["lr"]

        print("\n============================================================")
        print(f"Starting {stage_name} stage")
        print("============================================================")
        print(f"Stage episodes      = {n_episodes}")
        print(f"epsilon_start       = {epsilon:.3f}")
        print(f"epsilon_end         = {epsilon_end:.3f}")
        print(f"learning rate       = {lr:.6f}")

        # Match the original two-stage logic: model weights continue, but the
        # optimizer, target network, replay buffer, and epsilon schedule are
        # re-created for the fine-tuning call.
        target_net = DQN(state_dim, n_actions).to(device)
        target_net.load_state_dict(policy_net.state_dict())
        target_net.eval()

        optimizer = optim.Adam(policy_net.parameters(), lr=lr)
        replay_buffer = ReplayBuffer(buffer_size)

        stage_rewards = []

        for stage_episode in range(1, n_episodes + 1):
            global_episode += 1

            state = env.reset()
            total_reward = 0.0
            done = False

            while not done:
                action = select_action_epsilon_greedy(
                    model=policy_net,
                    state=state,
                    n_actions=n_actions,
                    epsilon=epsilon,
                    device=device
                )

                next_state, reward, done, info = env.step(action)

                replay_buffer.push(
                    state,
                    action,
                    reward,
                    next_state,
                    done
                )

                state = next_state
                total_reward += reward

                if len(replay_buffer) >= batch_size:
                    states, actions_b, rewards_b, next_states, dones = replay_buffer.sample(
                        batch_size
                    )

                    states_t = torch.tensor(states, dtype=torch.float32, device=device)
                    actions_t = torch.tensor(actions_b, dtype=torch.int64, device=device).unsqueeze(1)
                    rewards_t = torch.tensor(rewards_b, dtype=torch.float32, device=device).unsqueeze(1)
                    next_states_t = torch.tensor(next_states, dtype=torch.float32, device=device)
                    dones_t = torch.tensor(dones, dtype=torch.float32, device=device).unsqueeze(1)

                    q_values = policy_net(states_t).gather(1, actions_t)

                    with torch.no_grad():
                        next_q_values = target_net(next_states_t).max(dim=1, keepdim=True)[0]
                        target_q_values = rewards_t + gamma * next_q_values * (1.0 - dones_t)

                    loss = F.mse_loss(q_values, target_q_values)

                    optimizer.zero_grad()
                    loss.backward()
                    torch.nn.utils.clip_grad_norm_(policy_net.parameters(), 5.0)
                    optimizer.step()

            stage_rewards.append(total_reward)
            all_rewards.append(total_reward)

            epsilon = max(epsilon_end, epsilon * epsilon_decay)

            if stage_episode % target_update == 0:
                target_net.load_state_dict(policy_net.state_dict())

            if verbose and stage_episode % 100 == 0:
                print(
                    f"{stage_name} episode {stage_episode:5d} "
                    f"(global {global_episode:5d}) | "
                    f"epsilon={epsilon:.3f} | "
                    f"mean reward={np.mean(stage_rewards[-100:]):.3f}"
                )

            if stage_episode % check_interval == 0:
                print("\n============================================================")
                print(
                    f"Checkpoint validation after {stage_name} episode "
                    f"{stage_episode} (global episode {global_episode})"
                )
                print("============================================================")

                eval_model = DQN(state_dim, n_actions)
                eval_model.load_state_dict({
                    k: v.detach().cpu().clone()
                    for k, v in policy_net.state_dict().items()
                })
                eval_model.eval()
                
                _,ats0, avg_h0, avg_n0, sampling_rate0 = estimate_ic_performance_fast(
                    model=eval_model,
                    cfg=cfg,
                    actions=actions,
                    n_rep=ic_check_rep,
                    n_jobs=ic_check_jobs
                )

                ic_ok = check_dqn_ic_constraints(
                    ats0=ats0,
                    avg_h0=avg_h0,
                    avg_n0=avg_n0,
                    sampling_rate0=sampling_rate0,
                    cfg=cfg
                )

                ic_score = constraint_score(
                    cfg=cfg,
                    avg_h0=avg_h0,
                    avg_n0=avg_n0
                )

                if ic_score < best_constraint_score:
                    best_constraint_score = ic_score
                    best_constraint_stage = stage_name
                    best_constraint_episode = global_episode
                    best_constraint_state = {
                        k: v.detach().cpu().clone()
                        for k, v in policy_net.state_dict().items()
                    }
                    best_constraint_metrics = (
                        ats0,
                        avg_h0,
                        avg_n0,
                        sampling_rate0
                    )

                mean_val_ats1 = np.nan
                mean_val_arl1 = np.nan

                if ic_ok:
                    print("\nIC constraints passed. Evaluating OC validation performance.")

                    oc_validation_df = evaluate_dqn_chart_fast(
                        model=eval_model,
                        cfg=cfg,
                        actions=actions,
                        shifts=validation_shifts,
                        n_rep=oc_check_rep,
                        n_jobs=ic_check_jobs
                    )

                    mean_val_ats1 = float(oc_validation_df["ATS1"].mean())
                    mean_val_arl1 = float(oc_validation_df["ARL1"].mean())

                    print(f"\nCheckpoint global episode {global_episode}: mean validation ATS1 = {mean_val_ats1:.4f}")
                    print(f"Checkpoint global episode {global_episode}: mean validation ARL1 = {mean_val_arl1:.4f}")

                    if mean_val_ats1 < best_feasible_mean_ats1:
                        best_feasible_mean_ats1 = mean_val_ats1
                        best_feasible_mean_arl1 = mean_val_arl1
                        best_feasible_stage = stage_name
                        best_feasible_episode = global_episode
                        best_feasible_state = {
                            k: v.detach().cpu().clone()
                            for k, v in policy_net.state_dict().items()
                        }
                        best_feasible_metrics = (
                            ats0,
                            avg_h0,
                            avg_n0,
                            sampling_rate0
                        )

                        print(f"New best feasible checkpoint selected at global episode {global_episode}.")
                else:
                    print("\nIC constraints not satisfied. OC validation skipped for this checkpoint.")

                checkpoint_history.append({
                    "stage": stage_name,
                    "stage_episode": stage_episode,
                    "global_episode": global_episode,
                    "ATS0": ats0,
                    "avg_h0": avg_h0,
                    "avg_n0": avg_n0,
                    "sampling_rate0": sampling_rate0,
                    "ic_score": ic_score,
                    "ic_ok": ic_ok,
                    "mean_validation_ATS1": mean_val_ats1,
                    "mean_validation_ARL1": mean_val_arl1
                })

    selected_model = DQN(state_dim, n_actions)

    if best_feasible_state is not None:
        selected_model.load_state_dict(best_feasible_state)
        selected_episode = best_feasible_episode
        selected_stage = best_feasible_stage
        selected_reason = "best_feasible_mean_validation_ATS1"
        selected_metrics = best_feasible_metrics

        print("\n============================================================")
        print("Best feasible checkpoint selected")
        print("============================================================")
        print(f"Selected stage           = {selected_stage}")
        print(f"Selected global episode  = {selected_episode}")
        print(f"Mean validation ATS1     = {best_feasible_mean_ats1:.4f}")
        print(f"Mean validation ARL1     = {best_feasible_mean_arl1:.4f}")

    else:
        selected_model.load_state_dict(best_constraint_state)
        selected_episode = best_constraint_episode
        selected_stage = best_constraint_stage
        selected_reason = "closest_IC_constraints_no_feasible_checkpoint"
        selected_metrics = best_constraint_metrics

        print("\n============================================================")
        print("No feasible checkpoint found")
        print("============================================================")
        print(f"Closest IC checkpoint stage          = {selected_stage}")
        print(f"Closest IC checkpoint global episode = {selected_episode}")
        print("This policy must still pass the final IC check before use.")

    selected_model.eval()

    return (
        selected_model.cpu(),
        all_rewards,
        checkpoint_history,
        selected_episode,
        selected_stage,
        selected_reason,
        selected_metrics
    )


# ============================================================
# GREEDY POLICY SIMULATION
# ============================================================

def get_cpu_state_dict(model):
    return {
        k: v.detach().cpu().clone()
        for k, v in model.state_dict().items()
    }


def greedy_action_cpu(model, state):
    with torch.no_grad():
        state_t = torch.tensor(
            state,
            dtype=torch.float32
        ).unsqueeze(0)

        q_values = model(state_t)

        return int(torch.argmax(q_values, dim=1).item())


def simulate_dqn_run_cpu(
    model,
    cfg,
    actions,
    delta,
    rng,
    max_steps=None
):
    if max_steps is None:
        max_steps = cfg.max_steps

    sigma_z = math.sqrt(cfg.lam / (2.0 - cfg.lam))
    z = rng.normal(loc=0.0, scale=sigma_z)
    z_prev = z

    #prev_n = min(cfg.n_values)
    #prev_h = max(cfg.h_values)
    prev_n = cfg.target_avg_n
    prev_h = cfg.target_avg_h


    H, HW, W, sigma_z = ewma_limits(
        cfg.L,
        cfg.p0,
        cfg.lam
    )

    total_n = 0.0
    total_h = 0.0
    total_steps = 0

    for step in range(max_steps):
        state = make_state(
            z,
            z_prev,
            prev_n,
            prev_h,
            H,
            HW,
            cfg
        )

        action_index = greedy_action_cpu(model, state)
        n_t, h_t = actions[action_index]

        y_t = rng.normal(
            loc=math.sqrt(n_t) * delta,
            scale=1.0
        )

        z_prev = z
        z = update_ewma(z_prev, y_t, cfg.lam)

        total_n += n_t
        total_h += h_t
        total_steps += 1

        prev_n = n_t
        prev_h = h_t

        if abs(z) > H:
            break

    RL = total_steps
    ATS = total_h

    return {
        "RL": RL,
        "ATS": ATS,
        "total_n": total_n,
        "total_h": total_h,
        "avg_n": total_n / max(RL, 1),
        "avg_h": total_h / max(RL, 1),
        "sampling_rate": total_n / max(total_h, 1e-12)
    }


# ============================================================
# IC PERFORMANCE OF LEARNED DQN POLICY
# ============================================================

def dqn_ic_chunk(
    model_state_dict,
    state_dim,
    n_actions,
    cfg,
    actions,
    n_rep,
    seed
):
    torch.set_num_threads(1)

    rng = np.random.default_rng(seed)
    cfg_local = copy.deepcopy(cfg)

    model = DQN(state_dim, n_actions)
    model.load_state_dict(model_state_dict)
    model.eval()

    rls = np.zeros(n_rep)
    ats = np.zeros(n_rep)
    total_ns = np.zeros(n_rep)

    for i in range(n_rep):
        out = simulate_dqn_run_cpu(
            model=model,
            cfg=cfg_local,
            actions=actions,
            delta=0.0,
            rng=rng
        )

        rls[i] = out["RL"]
        ats[i] = out["ATS"]
        total_ns[i] = out["total_n"]

    return rls, ats, total_ns


def estimate_ic_performance_fast(
    model,
    cfg,
    actions,
    n_rep=100000,
    n_jobs=N_JOBS
):
    model_state_dict = get_cpu_state_dict(model)

    state_dim = 5
    n_actions = len(actions)

    chunks = split_reps(n_rep, n_jobs)

    results = Parallel(
        n_jobs=len(chunks),
        backend="loky"
    )(
        delayed(dqn_ic_chunk)(
            model_state_dict,
            state_dim,
            n_actions,
            cfg,
            actions,
            chunks[j],
            98765 + j
        )
        for j in range(len(chunks))
    )

    rls = np.concatenate([r[0] for r in results])
    ats = np.concatenate([r[1] for r in results])
    total_ns = np.concatenate([r[2] for r in results])

    arl0 = float(np.mean(rls))
    ats0 = float(np.mean(ats))

    total_steps = float(np.sum(rls))
    total_time = float(np.sum(ats))
    total_n = float(np.sum(total_ns))
    
    # Paper-consistent definitions
    ats0_over_arl0 = ats0 / arl0
    avg_h0 = ats0_over_arl0
    
    # Mean of per-run average sample size
    avg_n0 = float(np.mean(total_ns / rls))
    
    # Global sampling rate
    sampling_rate0 = total_n / total_time
    
    return arl0, ats0, avg_h0, avg_n0, sampling_rate0


# ============================================================
# OC PERFORMANCE OF LEARNED DQN POLICY
# ============================================================

def dqn_oc_chunk(
    model_state_dict,
    state_dim,
    n_actions,
    cfg,
    actions,
    delta,
    n_rep,
    seed
):
    torch.set_num_threads(1)

    rng = np.random.default_rng(seed)
    cfg_local = copy.deepcopy(cfg)

    model = DQN(state_dim, n_actions)
    model.load_state_dict(model_state_dict)
    model.eval()

    rls = np.zeros(n_rep)
    ats = np.zeros(n_rep)
    total_ns = np.zeros(n_rep)

    for i in range(n_rep):
        out = simulate_dqn_run_cpu(
            model=model,
            cfg=cfg_local,
            actions=actions,
            delta=delta,
            rng=rng
        )

        rls[i] = out["RL"]
        ats[i] = out["ATS"]
        total_ns[i] = out["total_n"]

    return rls, ats, total_ns


def evaluate_dqn_chart_fast(
    model,
    cfg,
    actions,
    shifts=(0.25, 0.5, 0.75, 1.0, 1.5, 2.0), #(0.01,0.05,0.1,0.15,0.25, 0.50, 0.75, 1.00, 1.25, 1.50, 1.75,2,3,5),
    n_rep=10000,
    n_jobs=N_JOBS
):
    model_state_dict = get_cpu_state_dict(model)

    state_dim = 5
    n_actions = len(actions)

    rows = []

    for delta in shifts:
        chunks = split_reps(n_rep, n_jobs)

        results = Parallel(
            n_jobs=len(chunks),
            backend="loky"
        )(
            delayed(dqn_oc_chunk)(
                model_state_dict,
                state_dim,
                n_actions,
                cfg,
                actions,
                delta,
                chunks[j],
                200000 + 1000 * int(delta * 100) + j
            )
            for j in range(len(chunks))
        )

        rls = np.concatenate([r[0] for r in results])
        ats = np.concatenate([r[1] for r in results])
        total_ns = np.concatenate([r[2] for r in results])

        arl1 = float(np.mean(rls))
        ats1 = float(np.mean(ats))

        total_steps = float(np.sum(rls))
        total_time = float(np.sum(ats))
        total_n = float(np.sum(total_ns))
        
        # Paper-consistent definitions
        avg_h = ats1 / arl1
        avg_n = float(np.mean(total_ns / rls))
        sampling_rate = total_n / total_time
        
        rows.append({
            "delta": delta,
            "ARL1": arl1,
            "ATS1": ats1,
            "SDRL1": float(np.std(rls, ddof=1)),
            "SDTS1": float(np.std(ats, ddof=1)),
            "avg_n": avg_n,
            "avg_h": avg_h,
            "sampling_rate": sampling_rate
        })

        print(f"Finished delta={delta}")

    return pd.DataFrame(rows)


# ============================================================
# FINAL CONSTRAINT CHECK
# ============================================================

def check_dqn_ic_constraints(
    ats0,
    avg_h0,
    avg_n0,
    sampling_rate0,
    cfg
):

    avg_h_ok = abs(avg_h0 - cfg.target_avg_h) <= cfg.avg_h_tol

    avg_n_ok = (
        abs(avg_n0 - cfg.target_avg_n) / cfg.target_avg_n
        <= cfg.avg_n_tol
    )

    print("\nDQN IC constraint check")

    print(f"target avg_h             = {cfg.target_avg_h:.4f}")
    print(f"estimated avg_h0         = {avg_h0:.4f}")
    print(f"avg_h ok                 = {avg_h_ok}")

    print(f"target avg_n             = {cfg.target_avg_n:.4f}")
    print(f"estimated avg_n0         = {avg_n0:.4f}")
    print(f"avg_n ok                 = {avg_n_ok}")

    print(f"sampling_rate0           = {sampling_rate0:.4f}")

    return avg_h_ok and avg_n_ok


def constraint_score(cfg, avg_h0, avg_n0):
    h_part = abs(avg_h0 - cfg.target_avg_h) / cfg.target_avg_h
    n_part = abs(avg_n0 - cfg.target_avg_n) / cfg.target_avg_n

    return h_part + n_part


# ============================================================
# POLICY BEHAVIOR SUMMARY
# ============================================================

def evaluate_policy_behavior(
    model,
    cfg,
    actions,
    n_rep=1000,
    max_steps=1000
):
    rng = np.random.default_rng(999)

    H, HW, W, sigma_z = ewma_limits(
        cfg.L,
        cfg.p0,
        cfg.lam
    )

    model_cpu = copy.deepcopy(model).cpu()
    model_cpu.eval()

    rows = []

    for rep in range(n_rep):
        z = rng.normal(loc=0.0, scale=sigma_z)   # sigma_z already computed above
        z_prev = z

        #prev_n = min(cfg.n_values)
        #prev_h = max(cfg.h_values)
        prev_n = cfg.target_avg_n
        prev_h = cfg.target_avg_h


        for step in range(max_steps):
            rho = min(abs(z) / H, 1.0)

            state = make_state(
                z,
                z_prev,
                prev_n,
                prev_h,
                H,
                HW,
                cfg
            )

            action_index = greedy_action_cpu(model_cpu, state)
            n_t, h_t = actions[action_index]

            rows.append({
                "rho": rho,
                "n_t": n_t,
                "h_t": h_t
            })

            y_t = rng.normal(loc=0.0, scale=1.0)

            z_prev = z
            z = update_ewma(z_prev, y_t, cfg.lam)

            prev_n = n_t
            prev_h = h_t

            if abs(z) > H:
                break

    policy_df = pd.DataFrame(rows)

    bins = [0.0, 0.25, 0.50, 0.75, 1.0]
    labels = ["0-0.25", "0.25-0.50", "0.50-0.75", "0.75-1.00"]

    policy_df["region"] = pd.cut(
        policy_df["rho"],
        bins=bins,
        labels=labels,
        include_lowest=True
    )

    policy_summary = policy_df.groupby("region", observed=False).agg(
        avg_n=("n_t", "mean"),
        avg_h=("h_t", "mean"),
        count=("n_t", "count")
    ).reset_index()

    print("\nPolicy behavior summary:")
    print(policy_summary)

    return policy_summary, policy_df


# ============================================================
# COMPLETE RUN FUNCTION
# ============================================================

def run_for_one_p0(
    p0=0.5,
    target_arl0=370.4,
    target_avg_n=3.5,
    initial_train_episodes=20000,
    fine_tune_episodes=10000,
    checkpoint_interval=1000,
    checkpoint_validation_rep=None,
    checkpoint_oc_rep=5000,
    checkpoint_validation_shifts=(0.25, 0.50, 0.75, 1.00, 1.50, 2.00),
    calibration_rep=10000,
    evaluation_rep=10000,
    enforce_final_ic_constraints=True,
    seed=12345
):
    set_global_seed(seed)

    cfg = Config()
    cfg.p0 = p0
    cfg.target_arl0 = target_arl0
    cfg.target_avg_n = target_avg_n

    validate_config(cfg)

    actions = create_actions(cfg)

    W = warning_coefficient_from_p0(cfg.p0)

    print("\n============================================================")
    print(f"Running revised DQN-VSSI-EWMA design for p0 = {p0}")
    print("============================================================")
    print(f"Warning coefficient W = {W:.5f}")
    print(f"Number of actions     = {len(actions)}")
    print(f"Target ARL0           = {cfg.target_arl0:.4f}")
    print(f"Target avg_h0         = {cfg.target_avg_h:.4f}")
    print(f"Target avg_n0         = {cfg.target_avg_n:.4f}")

    print("\nPre-training calibration of L")

    calibrated_L = calibrate_L_direct(
        cfg=cfg,
        target_arl0=cfg.target_arl0,
        low=None,
        high=5.0,
        n_iter=20,
        n_rep=calibration_rep
    )

    cfg.L = calibrated_L
    
    print("\nEstimating IC mean rho for structural target calibration")
    
    cfg.mean_rho_ic = estimate_direct_mean_rho_ic(
        L=cfg.L,
        cfg=cfg,
        n_rep=calibration_rep,
        n_jobs=N_JOBS
    )
    
    set_vssi_structural_targets_from_ic_rho(cfg)

    if checkpoint_validation_rep is None:
        checkpoint_validation_rep = calibration_rep

    print("\n============================================================")
    print("Two-stage DQN training with best-checkpoint selection")
    print("============================================================")
    print(f"Initial training episodes       = {initial_train_episodes}")
    print(f"Fine-tuning episodes            = {fine_tune_episodes}")
    print(f"Checkpoint interval             = {checkpoint_interval}")
    print(f"Checkpoint IC MC replications   = {checkpoint_validation_rep}")
    print(f"Checkpoint OC MC replications   = {checkpoint_oc_rep}")
    print(f"Checkpoint validation shifts    = {checkpoint_validation_shifts}")

    (
        model,
        all_rewards,
        checkpoint_history,
        selected_episode,
        selected_stage,
        selected_reason,
        selected_metrics
    ) = train_dqn_two_stage_best_checkpoint(
        cfg=cfg,
        actions=actions,
        initial_train_episodes=initial_train_episodes,
        fine_tune_episodes=fine_tune_episodes,
        check_interval=checkpoint_interval,
        ic_check_rep=checkpoint_validation_rep,
        oc_check_rep=checkpoint_oc_rep,
        validation_shifts=checkpoint_validation_shifts,
        ic_check_jobs=N_JOBS,
        gamma=0.995,
        batch_size=64,
        buffer_size=50000,
        target_update=100,
        verbose=True
    )

    checkpoint_df = pd.DataFrame(checkpoint_history)

    if len(checkpoint_df) > 0:
        checkpoint_df.to_csv(
            f"dqn_ic_checkpoint_history_p{str(p0).replace('.', '')}.csv",
            index=False
        )

    print("\n============================================================")
    print("Final selected DQN policy")
    print("============================================================")
    print(f"Selected checkpoint stage          = {selected_stage}")
    print(f"Selected checkpoint global episode = {selected_episode}")
    print(f"Selection reason                   = {selected_reason}")

    H, HW, W, sigma_z = ewma_limits(
        cfg.L,
        cfg.p0,
        cfg.lam
    )

    print(f"Final L              = {cfg.L:.5f}")
    print(f"Final W              = {W:.5f}")
    print(f"Final sigma_Z        = {sigma_z:.5f}")
    print(f"Final H              = {H:.5f}")
    print(f"Final HW             = {HW:.5f}")
    print(f"Final HW/H           = {W / cfg.L:.5f}")

    print("\nEstimating final IC performance")

    arl0, ats0, avg_h0, avg_n0, sampling_rate0 = estimate_ic_performance_fast(
        model=model,
        cfg=cfg,
        actions=actions,
        n_rep=calibration_rep
    )

    final_ok = check_dqn_ic_constraints(
        ats0=ats0,
        avg_h0=avg_h0,
        avg_n0=avg_n0,
        sampling_rate0=sampling_rate0,
        cfg=cfg
    )

    policy_summary, policy_df = evaluate_policy_behavior(
        model=model,
        cfg=cfg,
        actions=actions,
        n_rep=1000
    )

    if enforce_final_ic_constraints and (not final_ok):
        raise RuntimeError(
            "Final DQN policy failed IC constraints. "
            "Do not use this DQN policy for final comparison. "
            "The procedure is correct, but this trained policy is not acceptable."
        )

    print("\nEstimating final OC performance")

    results_df = evaluate_dqn_chart_fast(
        model=model,
        cfg=cfg,
        actions=actions,
        shifts=(0.01,0.05,0.1,0.15,0.25, 0.50, 0.75, 1.00, 1.25, 1.50, 1.75,2,3,5),
        n_rep=evaluation_rep
    )

    print("\nDQN OC results")
    print(results_df)

    results_df.to_csv(
        f"dqn_vssi_ewma_oc_results_p{str(p0).replace('.', '')}.csv",
        index=False
    )

    policy_summary.to_csv(
        f"dqn_policy_summary_p{str(p0).replace('.', '')}.csv",
        index=False
    )

    outputs = {
        "cfg": cfg,
        "model": model,
        "actions": actions,
        "L": cfg.L,
        "W": W,
        "H": H,
        "HW": HW,
        "sigma_z": sigma_z,
        "ARL0": arl0,
        "ATS0": ats0,
        "ATS0_over_ARL0": avg_h0,
        "avg_h0": avg_h0,
        "avg_n0": avg_n0,
        "sampling_rate0": sampling_rate0,
        "selected_episode": selected_episode,
        "selected_stage": selected_stage,
        "selected_reason": selected_reason,
        "checkpoint_history": checkpoint_df,
        "results_df": results_df if final_ok else None,
        "policy_summary": policy_summary,
        "policy_df": policy_df,
        "rewards": all_rewards
    }

    return outputs


# ============================================================
outputs = run_for_one_p0(
    p0=0.5,
    target_arl0=370.4,
    target_avg_n=3.5,
    initial_train_episodes=20000,
    fine_tune_episodes=10000,
    checkpoint_interval=1000,
    checkpoint_validation_rep=100000,
    checkpoint_oc_rep=10000,
    checkpoint_validation_shifts=(0.25, 0.50, 0.75, 1.00, 1.50, 2.00),
    calibration_rep=100000,
    evaluation_rep=100000,
    enforce_final_ic_constraints=True,
    seed=12345
)
elapsed = time.perf_counter() - notebook_start
print(f"Total notebook execution time: {elapsed/60:.2f} minutes")

In [ ]:

# ============================================================
# Cell 2: This cell creates checkpoint-validation figure for the general case
# ============================================================

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

history = outputs["checkpoint_history"].copy()

if not isinstance(history, pd.DataFrame):
    history = pd.DataFrame(history)

history = history.sort_values("global_episode").reset_index(drop=True)

required_columns = {
    "global_episode",
    "ic_ok",
    "mean_validation_ATS1",
    "avg_h0",
    "avg_n0",
}
missing = required_columns.difference(history.columns)

if missing:
    raise KeyError(
        f"Missing checkpoint-history columns: {sorted(missing)}"
    )

selected_episode = int(outputs["selected_episode"])

selected_rows = history.loc[
    np.isclose(history["global_episode"], selected_episode)
]

if selected_rows.empty:
    raise ValueError(
        "The selected episode was not found in checkpoint_history."
    )

selected_row = selected_rows.iloc[0]

# Only feasible checkpoints receive out-of-control validation.
feasible = history.loc[
    history["ic_ok"].astype(bool)
    & history["mean_validation_ATS1"].notna()
].copy()

if feasible.empty:
    raise ValueError("No feasible checkpoints with validation ATS1 were found.")

fig, axes = plt.subplots(
    nrows=3,
    ncols=1,
    figsize=(8, 10),
    sharex=True,
)

# ---------------------------------------------------------
# Panel 1: validation ATS1 for feasible checkpoints
# ---------------------------------------------------------
axes[0].plot(
    feasible["global_episode"],
    feasible["mean_validation_ATS1"],
    marker="o",
    linewidth=2,
    label="Feasible checkpoint",
)

axes[0].scatter(
    [selected_episode],
    [float(selected_row["mean_validation_ATS1"])],
    marker="*",
    s=180,
    zorder=5,
    label=f"Selected checkpoint: episode {selected_episode}",
)

axes[0].set_ylabel(r"Mean validation $ATS_1$")
axes[0].set_title(
    "Checkpoint validation and in-control feasibility of the general DQN"
)
axes[0].legend()

# ---------------------------------------------------------
# Panel 2: average sampling interval
# ---------------------------------------------------------
axes[1].plot(
    history["global_episode"],
    history["avg_h0"],
    marker="o",
    linewidth=2,
)

axes[1].axhline(1.00, linestyle="-", linewidth=2.0, label="Target")
axes[1].axhline(0.97, linestyle=":", linewidth=2.0, label="Acceptance bounds")
axes[1].axhline(1.03, linestyle=":", linewidth=2.0)

axes[1].scatter(
    [selected_episode],
    [float(selected_row["avg_h0"])],
    marker="*",
    s=180,
    zorder=5,
)

axes[1].set_ylabel(r"Estimated $\bar h_0$")
axes[1].legend()

# ---------------------------------------------------------
# Panel 3: average sample size
# ---------------------------------------------------------
n_target = 3.5
n_lower = n_target * (1.0 - 0.03)
n_upper = n_target * (1.0 + 0.03)

axes[2].plot(
    history["global_episode"],
    history["avg_n0"],
    marker="o",
    linewidth=2,
)

axes[2].axhline(n_target, linestyle="-", linewidth=2.0, label="Target")
axes[2].axhline(n_lower, linestyle=":", linewidth=2.0, label="Acceptance bounds")
axes[2].axhline(n_upper, linestyle=":", linewidth=2.0)

axes[2].scatter(
    [selected_episode],
    [float(selected_row["avg_n0"])],
    marker="*",
    s=180,
    zorder=5,
)

axes[2].set_xlabel("Global training episode")
axes[2].set_ylabel(r"Estimated $\bar n_0$")
axes[2].legend()

# Boundary between initial training and fine-tuning.

plt.tight_layout()


for ax in axes:
    ax.axvline(
        20000,
        linestyle="--",
        linewidth=2,
        label="Fine-tuning begins"
    )
    ax.grid(alpha=0.25)

for ax in axes:
    ax.legend()

plt.savefig(
    "checkpoint_validation_general_DQN.png",
    dpi=600,
    bbox_inches="tight",
)
plt.show()

In [ ]:

# ========================================================================================================================
# Cell 3: This cell creates the Illustrative Examples (Phase II monitoring: change delta_oc value for a desired shift size)
# ========================================================================================================================

import numpy as np
import pandas as pd
import torch
import math
from statistics import NormalDist

# ==================================================
# Extract trained general DQN, config and actions
# ==================================================
model   = outputs["model"]
cfg     = outputs["cfg"]
actions = outputs["actions"]

model.eval()
model_cpu = model.cpu()

def greedy_action(state):
    with torch.no_grad():
        state_t = torch.tensor(state, dtype=torch.float32).unsqueeze(0)
        q_values = model_cpu(state_t)
        return int(torch.argmax(q_values, dim=1).item())

# EWMA and chart parameters
lam   = cfg.lam
L_val = cfg.L
p0    = cfg.p0

sigma_z = math.sqrt(lam / (2.0 - lam))
W = NormalDist().inv_cdf((1.0 + p0) / 2.0)   # warning coefficient
H = L_val * sigma_z        # control limit
HW = W * sigma_z           # warning limit

n_max = max(cfg.n_values)   # 10
h_max = max(cfg.h_values)   # 2.0

# Initial state
lam=0.2
sigma_z = math.sqrt(lam / (2.0 - lam))
z = np.random.normal(0.0, sigma_z)
z_prev = z
prev_n = cfg.target_avg_n
prev_h = cfg.target_avg_h

rows = []
shift_start = 8          # step at which the shift occurs
delta_oc   = 3        # out‑of‑control shift size 

for step in range(1, 21):          # simulate 20 steps (or break if signal)
    # Determine true process state
    if step < shift_start:
        delta = 0.0
        true_state = "IC"
    else:
        delta = delta_oc
        true_state = "OC"

    # Build state vector (same 5 dimensions as in training)
    closeness = abs(z) / H
    movement = abs(z - z_prev) / H
    prev_n_scaled = prev_n / n_max
    prev_h_scaled = prev_h / h_max
    warning_ind = 1.0 if abs(z) > HW else 0.0
    state = np.array([closeness, movement, prev_n_scaled, prev_h_scaled, warning_ind],
                     dtype=np.float32)

    # DQN decision
    action_idx = greedy_action(state)
    n_t, h_t = actions[action_idx]

    # Sample observation from the true process distribution
    y_t = np.random.normal(loc=math.sqrt(n_t) * delta, scale=1.0)

    # Update EWMA
    z_prev = z
    z = lam * y_t + (1.0 - lam) * z_prev

    # Chart decision
    signal = abs(z) > H

    rows.append({
        "Step": step,
        "True state": true_state,
        "Z_t": f"{z:.3f}",
        "closeness": f"{closeness:.3f}",
        "movement": f"{movement:.3f}",
        "prev_n_scaled": f"{prev_n_scaled:.3f}",
        "prev_h_scaled": f"{prev_h_scaled:.3f}",
        "warning": int(warning_ind),
        "Action (n,h)": f"({n_t}, {h_t:.2f})",
        "Y_t": f"{y_t:.3f}",
        "Signal": "Yes" if signal else "No"
    })

    if signal:
        break

    # Prepare previous action for next state
    prev_n = n_t
    prev_h = h_t

# Display the online monitoring table
df = pd.DataFrame(rows)
print(df.to_string(index=False))